## Imports

In [55]:
import scipy.io
import numpy as np
from scipy.stats import linregress
from statsmodels.sandbox.stats.multicomp import multipletests 

# seaborn can be used to "prettify" default matplotlib plots by importing and setting as default
import seaborn as sns
sns.set() # Set searborn as default

## Load dataset

In [56]:
mat = scipy.io.loadmat('sand.mat')
X = mat['X']
y = mat['Y'].ravel()

[n, p] = X.shape

### 3 Perform univariate feature selection for the sand data using:

> (a) Bonferroni correction to control the family-wise error rate(FWER). Use FWER = 0.05.

In [57]:

# Sort p-values in acending order

# include all features with p values lower than p / features

FWER = 0.05
PValues = np.zeros(p)
Xsub = np.zeros(p)

for j in range(p):
# Calculate the pvalue for each feature one at the time because OLS breaks down with this many features
# Use the stats models linear regression, since p value already is included
# Otherwise check https://stackoverflow.com/questions/27928275/find-p-value-significance-in-scikit-learn-linearregression
# Which explains how to expand the class in sklearn to calculate it

  Xsub = X[:,j]
  slope, intercept, r_value, PValues[j], std_err = linregress(Xsub, y)

idx1 = np.argsort(PValues)
p = PValues[idx1]

print(p)
features = len(p)
print(features)

remaining_features_bonf = len(np.where(p < (0.05 / features))[0]) # Amount af features included
print(f'Remaining features after correcting with Bonferroni correction: {remaining_features_bonf}.')


[2.44459409e-12 2.60901450e-12 1.69771434e-11 ... 9.95521660e-01
 9.98049600e-01 9.99068164e-01]
2016
Remaining features after correcting with Bonferroni correction: 72.


> (b) Benjamini-Hochberg’s algorithm for FDR. Use an acceptable fraction of mistakes,
q = 0.15.

In [59]:
# Use multipletests to get the FDR corrected p values

# Sort p-values in acending order

# include all features with p values lower  than q

q = 0.15

FDR = multipletests(PValues, alpha = 0.05, method = "fdr_bh")[1] # Computing Benjamini Hochberg's FDR

idx2 = np.argsort(FDR)
fdr = FDR[idx2]

k = 0
for i in range(len(fdr)):
  if fdr[i] < q:
    print(f'Feature {i+1} is rejected, {fdr[i]} < {q}')
  else:
    k = i
    print(f'Feature {i+1}')

print(f'Remaining features after correcting with Benjamini-Hochberg correction:  {len(np.where(fdr < 0.15)[0])}.')


Feature 1 is rejected, 2.6298866163522346e-09 < 0.15
Feature 2 is rejected, 2.6298866163522346e-09 < 0.15
Feature 3 is rejected, 1.1408640364006537e-08 < 0.15
Feature 4 is rejected, 1.153882744374995e-08 < 0.15
Feature 5 is rejected, 5.394011385855236e-08 < 0.15
Feature 6 is rejected, 1.0218899194208812e-07 < 0.15
Feature 7 is rejected, 1.8734390622660777e-07 < 0.15
Feature 8 is rejected, 2.2831839805739472e-07 < 0.15
Feature 9 is rejected, 2.2831839805739472e-07 < 0.15
Feature 10 is rejected, 2.2831839805739472e-07 < 0.15
Feature 11 is rejected, 2.680919614772315e-07 < 0.15
Feature 12 is rejected, 1.918694613052869e-06 < 0.15
Feature 13 is rejected, 3.179842908702336e-06 < 0.15
Feature 14 is rejected, 3.225648945921439e-06 < 0.15
Feature 15 is rejected, 3.450951128073675e-06 < 0.15
Feature 16 is rejected, 4.354799317597238e-06 < 0.15
Feature 17 is rejected, 4.909397886498885e-06 < 0.15
Feature 18 is rejected, 8.94600323900478e-06 < 0.15
Feature 19 is rejected, 1.0780789836879582e-05 <

Compare the solutions in terms of number of selected features and selected features.

*It is clear that FDR "allows" for more features to be kept in the model, and through this the chance of having false discoveries are higher, this is done to make sure that all significant features are kept in the model, whereas bonferroni might remove some significant features because of the more stringent cutoff.*